<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-10-production-deployment/lesson-10.5-docker-and-containerization/notebooks/exercises-10.5.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 10.5 — Docker & Containerization
Netsetos GenAI Course v2.0 | Module 10: Production Deployment

Multi-stage builds, base images, CUDA containers, BuildKit, distroless, Trivy, cosign, docker-compose local stack.


In [ ]:
print('Module 10: Production Deployment')
print('Lesson 10.5: Docker & Containerization')


## Ex 1: Base Image Comparison


In [ ]:
print('Base image choices for AI apps:')
for base, size, best_for in [
    ('python:3.12-slim','~120 MB','Default for AI apps (glibc, wheels work)'),
    ('python:3.12-alpine','~55 MB','AVOID - musl breaks numpy/torch wheels'),
    ('gcr.io/distroless/python3-debian12','~50 MB','Production - no shell, minimal CVEs'),
    ('nvidia/cuda:12.4.0-runtime-ubuntu22.04','~3 GB','GPU inference (match torch cuda)'),
    ('nvidia/cuda:12.4.0-devel','~6 GB','NEVER in prod - nvcc not needed'),
]:
    print(f'  {base:40s}: {size:8s} | {best_for}')


## Ex 2: Image Size Progression


In [ ]:
print('Typical size progression for a FastAPI + RAG app:')
for step, size_mb, delta in [
    ('naive single-stage',    1500, 'starting point'),
    ('+ .dockerignore',       1480, '- 20 MB (context only, not image)'),
    ('+ multi-stage',          320, '-79% vs naive'),
    ('+ no-cache-dir pip',     278, '-13% more'),
    ('+ distroless runtime',    94, '-66% vs slim runtime'),
]:
    print(f'  {step:28s} -> {size_mb:>5} MB   ({delta})')


## Ex 3: Build Optimization Techniques


In [ ]:
print('Build optimization impact:')
for technique, impact, when in [
    ('Multi-stage build','1.5 GB -> 280 MB','Always'),
    ('BuildKit cache mounts','4 min -> 25 s rebuild','When deps change rarely'),
    ('.dockerignore','GB context -> MB','Always'),
    ('Layer order: deps before code','High cache hit on rebuild','Always'),
    ('--no-cache-dir on pip','~50 MB saved','Always in runtime stage'),
    ('Combined RUN apt-get + rm','~30 MB saved','Always'),
]:
    print(f'  {technique:32s}: {impact:24s} | When: {when}')


## Ex 4: Container Runtime Comparison


In [ ]:
print('Container runtimes:')
for rt, strength, best_for in [
    ('Docker','Ubiquitous, great DX','Local dev, CI'),
    ('Podman','Daemonless, rootless','Security-sensitive hosts'),
    ('containerd','K8s default runtime','Production K8s clusters'),
    ('CRI-O','Lightweight K8s runtime','OpenShift / RHEL clusters'),
]:
    print(f'  {rt:14s}: {strength:28s} | Best: {best_for}')


## Ex 5: Supply Chain Security


In [ ]:
print('Supply chain checklist:')
for control, tool, notes in [
    ('Pin base image by digest','FROM img@sha256:...','Prevents tag drift'),
    ('Scan for CVEs','Trivy / Snyk','Fail CI on HIGH/CRITICAL'),
    ('SBOM generation','Syft','Compliance + audit'),
    ('Image signing','cosign (keyless OIDC)','Verify on pull'),
    ('Attestation','cosign attest','SLSA provenance'),
    ('Registry trust','Artifact Registry IAM','Limit who can push'),
]:
    print(f'  {control:24s}: {tool:18s} | {notes}')


## Ex 6: Deployment Target Decision


In [ ]:
print('Where to deploy your AI container:')
for target, model_fit, cold_start, best_for in [
    ('Cloud Run (CPU)','API-calling apps','~1 s','MVPs, bursty traffic'),
    ('Cloud Run (L4 GPU)','Models <= 13B','~5 s','Small self-hosted'),
    ('GKE','70B+, multi-GPU','N/A (always on)','Sustained QPS > 10'),
    ('AWS Lambda','No GPU','~500 ms','Orchestration only'),
    ('E2E / Modal','Spot GPU','depends','Cost-sensitive India'),
]:
    print(f'  {target:20s}: models={model_fit:18s} | cold={cold_start:8s} | {best_for}')


## Ex 7: docker-compose Local Prod-Like Stack


In [ ]:
print('Local stack services (docker-compose):')
for svc, role in [
    ('app','FastAPI or Next.js frontend'),
    ('litellm','Gateway proxy (10.3)'),
    ('redis','Cache + rate limits'),
    ('postgres+pgvector','Memory / RAG / keys'),
    ('langfuse','Observability (10.8)'),
    ('prometheus + grafana','Metrics dashboards'),
]:
    print(f'  {svc:22s}: {role}')
print()
print('Mirror prod env vars via .env.local so compose and Cloud Run share shape.')


---
## Recap
Multi-stage + slim/distroless + BuildKit cache + .dockerignore = 280 MB images, fast pulls.
Avoid alpine for AI (musl breaks wheels). Use CUDA runtime (not devel) in prod.
Trivy + cosign keyless + SBOM for supply chain.
Cloud Run for MVPs, GKE for 70B+. docker-compose mirrors prod for local dev.
